In [ ]:
import os 
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import torch.nn as nn
from torch.nn import functional as F
from torch import optim
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from transformers import Wav2Vec2CTCTokenizer,get_cosine_schedule_with_warmup
from jiwer import wer 
import torchaudio

In [ ]:
class SpeechDataset(Dataset):
    def __init__(self, data_dir, include_splits= ["train-clean-100", "train-clean-360", "train-clean-500"],sampling_rate=16000, num_audio_channels=1):
        
        if isinstance(include_splits, str):
            include_splits = [include_splits]
            
        self.sampling_rate = sampling_rate
        self.num_audio_channels = num_audio_channels
        
        # path to audio files
        self.librispeech_data = []
        for split in include_splits:
            split_path = os.path.join(data_dir, split)
            
            for speaker in os.listdir(split_path):
                path_to_speaker = os.path.join(split_path, speaker)
                
                
                for section in os.listdir(split_path):
                    path_to_section = os.path.join(path_to_speaker, section)
                    
                    # split flac audio and text transcripts
                    files = os.listdir(path_to_section)
                    transcript_file = [path for path in files if ".txt" in path[0]]
                    
                    # load transcript 
                    with open(os.path.join(path_to_section, transcript_file), "r") as f:
                        transcripts = f.readlines()
                        
                    # split transcript by audio filename 
                    for line in transcripts:
                        split_line = line.split()
                        audio_root = split_line[0]
                        audio_file = audio_root + ".flac"
                        full_path_to_audiofile = os.path.join(path_to_section, audio_file)
                        transcript = " ".join(split_line[1:]).strip()
                        
                        self.librispeech_data.append((full_path_to_audiofile, transcript))
                        
                        
        self.audio_to_mels = T.MelSpectrogram(sample_rate=self.sampling_rate, n_mels=80)
        self.amp_to_db = T.AmplitudeToDB(top_db=80.0)
        
    def __len__(self):
        return len(self.librispeech_data)
    
    def __getitem__(self, idx):
        
        # path to audio and transcript
        audio_path , transcript = self.librispeech_data[idx]
        
        audio,original_sr = torchaudio.load(audio_path, normalize=True)